In [1]:
# 1. Connect Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# 3. Paths
PROJECT_DIR = '/content/drive/MyDrive/ECG_Vision_App'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
IMG_DIR = '/content/ecg_images'
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 4. Restore the ECG images from Drive
!unzip -q -n {DATA_DIR}/ecg_images.zip -d /content

# 5. Load labels
data = pd.read_csv(os.path.join(DATA_DIR, 'af_vs_normal_labels.csv'), index_col='ecg_id')
print(data['split'].value_counts())
print("Images available:", len(os.listdir(IMG_DIR)))

# 6. Check the GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using:", device)
if device == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
split
train    7759
val       992
test      967
Name: count, dtype: int64
Images available: 9718
Using: cuda
GPU: Tesla T4


In [2]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 1. How every image is prepared
IMG_SIZE = (240, 496)   # (height, width): keeps the ECG's wide shape

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# 2. A Dataset: how to fetch ONE image and its label
class ECGImageDataset(Dataset):
    def __init__(self, table):
        self.ids = table.index.tolist()
        self.labels = table['label'].tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img = Image.open(f'{IMG_DIR}/{self.ids[i]}.png').convert('RGB')
        label = torch.tensor(self.labels[i], dtype=torch.float32)
        return transform(img), label

train_ds = ECGImageDataset(data[data['split'] == 'train'])
val_ds   = ECGImageDataset(data[data['split'] == 'val'])
test_ds  = ECGImageDataset(data[data['split'] == 'test'])

# 3. DataLoaders: feed the images in batches of 32
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

# 4. Check it works
print("Images per split:", len(train_ds), len(val_ds), len(test_ds))
images, labels = next(iter(train_loader))
print("One batch of images:", images.shape)
print("One batch of labels:", labels.shape, labels[:10])

Images per split: 7759 992 967
One batch of images: torch.Size([32, 3, 240, 496])
One batch of labels: torch.Size([32]) tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0.])


In [3]:
import torch.nn as nn
from torchvision import models

# 1. The model: a pretrained ResNet-18, with a new final layer for 1 output
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

# 2. The loss function, with extra weight on AF mistakes
train_labels = data[data['split'] == 'train']['label']
n_normal = (train_labels == 0).sum()
n_af = (train_labels == 1).sum()
pos_weight = torch.tensor([n_normal / n_af], device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# 3. The optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 4. Check it works on one batch (before any training)
print(f"Training set: {n_normal} normal, {n_af} AF → AF weight = {pos_weight.item():.2f}")
model.eval()
with torch.no_grad():
    out = model(images.to(device))
print("Model output shape:", out.shape)
print("First 5 AF probabilities (untrained):",
      torch.sigmoid(out[:5]).squeeze().cpu().numpy().round(2))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 203MB/s]


Training set: 6548 normal, 1211 AF → AF weight = 5.41
Model output shape: torch.Size([32, 1])
First 5 AF probabilities (untrained): [0.52 0.53 0.47 0.52 0.49]


In [5]:
# ---------- Imports ----------
# sklearn (scikit-learn): a standard machine learning library.
# f1_score     -> calculates the F1 score (balance of catching AF vs. false alarms)
# recall_score -> calculates sensitivity (and, with pos_label=0, specificity)
from sklearn.metrics import f1_score, recall_score


# ---------- Function 1: train the model for ONE epoch ----------
# One epoch = one full pass through all 7,759 training images.
def train_one_epoch():
    model.train()                                   # switch the model into learning mode
    total_loss = 0                                  # running total of the loss, to report the average

    for images, labels in train_loader:             # take the training images 32 at a time
        images = images.to(device)                  # move the images onto the GPU
        labels = labels.to(device)                  # move the labels onto the GPU (same device as the model)

        optimizer.zero_grad()                       # 1. clear the adjustments from the previous batch
        logits = model(images).squeeze(1)           # 2. forward pass: raw AF scores, shape [32, 1] -> [32]
        loss = loss_fn(logits, labels)              # 3. loss: how wrong the scores are (AF mistakes weighted 5.41x)
        loss.backward()                             # 4. backpropagation: work out how each weight caused the error
        optimizer.step()                            # 5. update: Adam nudges every weight to reduce the error

        total_loss += loss.item() * len(labels)     # add this batch's loss (scaled by batch size) to the total

    return total_loss / len(train_loader.dataset)   # average loss per image for this epoch


# ---------- Function 2: get predictions WITHOUT learning ----------
# Used for the validation and test sets.
def predict(loader):
    model.eval()                                    # switch the model into testing mode
    probs, all_labels = [], []                      # lists to collect the results

    with torch.no_grad():                           # no learning here -> skip the backpropagation bookkeeping
        for images, labels in loader:               # take the images 32 at a time
            logits = model(images.to(device)).squeeze(1)   # raw AF scores from the model
            probs.append(torch.sigmoid(logits).cpu())      # convert to probabilities (0-1), move back to the CPU
            all_labels.append(labels)                      # keep the true labels in the same order

    probs = torch.cat(probs).numpy()                       # join all batches into one list of probabilities
    all_labels = torch.cat(all_labels).numpy().astype(int) # join all labels, as whole numbers (0 or 1)
    return probs, all_labels

In [6]:
# ---------- Imports ----------
# time: Python's built-in clock, used to measure how long each epoch takes
import time


# ---------- Settings ----------
EPOCHS = 5          # how many full passes through the training images
best_f1 = -1        # best validation F1 so far (starts below any real score)


# ---------- The training loop ----------
for epoch in range(1, EPOCHS + 1):                  # epoch = 1, 2, 3, 4, 5
    start = time.time()                             # note the start time

    # 1. LEARN from the 7,759 training images
    train_loss = train_one_epoch()

    # 2. CHECK on the 992 validation images (no learning)
    val_probs, val_labels = predict(val_loader)     # AF probabilities + true labels
    val_pred = (val_probs > 0.5).astype(int)        # threshold 0.5 for now: above -> AF (1), else Normal (0)

    f1 = f1_score(val_labels, val_pred)                      # F1 score
    sens = recall_score(val_labels, val_pred)                # sensitivity: share of real AF caught
    spec = recall_score(val_labels, val_pred, pos_label=0)   # specificity: share of real normals cleared

    # 3. REPORT this epoch's results
    print(f"Epoch {epoch}: train loss {train_loss:.3f} | val F1 {f1:.3f} | "
          f"sensitivity {sens:.3f} | specificity {spec:.3f} | {time.time() - start:.0f}s")

    # 4. SAVE the model to Drive if this is the best validation F1 so far
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(),                        # the model's learned weights
                   os.path.join(MODEL_DIR, 'resnet18_best.pt'))  # saved to Drive: ECG_Vision_App/models/
        print("   ✓ new best model saved to Drive")

Epoch 1: train loss 0.174 | val F1 0.976 | sensitivity 0.960 | specificity 0.999 | 131s
   ✓ new best model saved to Drive
Epoch 2: train loss 0.051 | val F1 0.976 | sensitivity 0.954 | specificity 1.000 | 128s
Epoch 3: train loss 0.028 | val F1 0.977 | sensitivity 0.967 | specificity 0.998 | 130s
   ✓ new best model saved to Drive
Epoch 4: train loss 0.030 | val F1 0.967 | sensitivity 0.967 | specificity 0.994 | 130s
Epoch 5: train loss 0.024 | val F1 0.976 | sensitivity 0.960 | specificity 0.999 | 129s


In [7]:
# confusion_matrix counts true negatives, false positives, false negatives and true positives
from sklearn.metrics import confusion_matrix

# Load the saved epoch-3 weights (the best model), because the model in memory is now from epoch 5
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'resnet18_best.pt'), map_location=device))

# Get the AF probabilities for every validation ECG (the predict function from Cell 4)
val_probs, val_labels = predict(val_loader)

# Turn the probabilities into 0/1 using the 0.5 threshold
val_preds = (val_probs >= 0.5).astype(int)

# Count the four outcomes and print them
tn, fp, fn, tp = confusion_matrix(val_labels, val_preds).ravel()
print(f'TN {tn} | FP {fp} | FN {fn} | TP {tp}')

# List the ECG IDs the model got wrong, so we can look at those images next
val_ids = np.array(val_ds.ids)
print('Missed AF (false negatives):', val_ids[(val_labels == 1) & (val_preds == 0)])
print('False alarms (false positives):', val_ids[(val_labels == 0) & (val_preds == 1)])

TN 839 | FP 2 | FN 5 | TP 146
Missed AF (false negatives): [ 7191 10919 11921 18526 19802]
False alarms (false positives): [ 1321 12171]


In [ ]:
# zipfile lets us read one file from inside the PTB-XL zip without unzipping everything
import zipfile
# ast turns the scp_codes text, e.g. "{'AFIB': 100.0}", back into a Python dictionary
import ast

# Find ptbxl_database.csv inside the zip on Drive and load it (it holds the cardiologists' labels)
with zipfile.ZipFile(f'{DATA_DIR}/ptb-xl-100hz.zip') as z:
    db_name = [n for n in z.namelist() if n.endswith('ptbxl_database.csv')][0]
    with z.open(db_name) as f:
        db = pd.read_csv(f, index_col='ecg_id')

# The 7 mistakes: first the 5 missed AF, then the 2 false alarms
wrong_ids = [7191, 10919, 11921, 18526, 19802, 1321, 12171]

# Map each validation ECG ID to the model's AF probability
prob_by_id = dict(zip(val_ids, val_probs))

# Show each mistake: the image, the true label, the model's probability and the original codes
for ecg_id in wrong_ids:
    true_label = 'AF' if data.loc[ecg_id, 'label'] == 1 else 'Normal'
    codes = ast.literal_eval(db.loc[ecg_id, 'scp_codes'])
    img = Image.open(f'{IMG_DIR}/{ecg_id}.png')
    plt.figure(figsize=(14, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'ECG {ecg_id} | true: {true_label} | model AF probability: {prob_by_id[ecg_id]:.2f}\n'
              f'codes: {codes} | report: {db.loc[ecg_id, "report"]}')
    plt.show()

In [9]:
# f1_score, recall_score and confusion_matrix measure how good each threshold is
from sklearn.metrics import f1_score, recall_score, confusion_matrix

# The thresholds to try: 0.05, 0.10, 0.15 ... 0.95
thresholds = np.arange(0.05, 1.0, 0.05)

# For each threshold, turn the validation probabilities into 0/1 and score the result
rows = []
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_labels, preds).ravel()
    rows.append({'threshold': round(t, 2),
                 'missed_AF': fn,                       # AF called normal (the dangerous error)
                 'false_alarms': fp,                    # normal called AF
                 'sensitivity': round(recall_score(val_labels, preds), 3),
                 'specificity': round(recall_score(val_labels, preds, pos_label=0), 3),
                 'F1': round(f1_score(val_labels, preds), 3)})

# Show everything as a table
results = pd.DataFrame(rows)
print(results.to_string(index=False))

# The threshold with the highest validation F1
best = results.loc[results['F1'].idxmax()]
print('\nBest threshold by F1:', best['threshold'])

 threshold  missed_AF  false_alarms  sensitivity  specificity    F1
      0.05          1            25        0.993        0.970 0.920
      0.10          1            20        0.993        0.976 0.935
      0.15          2            14        0.987        0.983 0.949
      0.20          2            10        0.987        0.988 0.961
      0.25          3             9        0.980        0.989 0.961
      0.30          3             6        0.980        0.993 0.970
      0.35          4             6        0.974        0.993 0.967
      0.40          4             6        0.974        0.993 0.967
      0.45          4             4        0.974        0.995 0.974
      0.50          5             2        0.967        0.998 0.977
      0.55          5             2        0.967        0.998 0.977
      0.60          5             1        0.967        0.999 0.980
      0.65          5             1        0.967        0.999 0.980
      0.70          6             1        0.960

In [10]:
# confusion_matrix, f1_score and recall_score score the final predictions
from sklearn.metrics import confusion_matrix, f1_score, recall_score

# The threshold you chose on the validation set; don't change it after seeing the test results
THRESHOLD = 0.30

# Get AF probabilities for the 967 test ECGs (fold 10), which the model has never seen
test_probs, test_labels = predict(test_loader)

# Turn the probabilities into 0/1 with your chosen threshold
test_preds = (test_probs >= THRESHOLD).astype(int)

# Count the four outcomes
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds).ravel()
print(f'TN {tn} | FP {fp} | FN {fn} | TP {tp}')

# The final scores you will report
print('Sensitivity:', round(recall_score(test_labels, test_preds), 3))
print('Specificity:', round(recall_score(test_labels, test_preds, pos_label=0), 3))
print('F1:', round(f1_score(test_labels, test_preds), 3))
print('Accuracy:', round((tp + tn) / len(test_labels), 3))

TN 811 | FP 4 | FN 6 | TP 146
Sensitivity: 0.961
Specificity: 0.995
F1: 0.967
Accuracy: 0.99
